# 01 — Baseline Face Verification & Presence Experiment (YuNet + SFace)

**Objective**: Perform controlled baseline validation of the face AI stack using OpenCV's official YuNet (Face Detection) and SFace (Face Recognition/Verification) models on a sample identity benchmark.

## 0. Colab Environment Initialization (Run First in Google Colab)
> **Note**: Google Colab comes with standard `opencv-python` pre-installed, which conflicts with `opencv-contrib-python`. Run the cell below to ensure clean package resolution and model acquisition.

In [1]:
# Step 0: Ensure clean opencv-contrib-python installation in Google Colab
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("Running in Google Colab. Initializing environment...")
    !pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless > /dev/null 2>&1
    !pip install -q "opencv-contrib-python>=4.8.0" "numpy>=1.24.0" pytest requests
    
    # Download official ONNX models with validation
    !python ../scripts/download_models.py || python scripts/download_models.py
    print("Colab environment setup complete!")
else:
    print("Running in local environment.")

Running in local environment.


## 1. Environment Setup & Model Initialization

In [2]:
import sys
from pathlib import Path
import cv2
import numpy as np

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().joinpath("src").exists() else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.face import FaceDetector, FaceVerifier, FacePresenceAnalyzer

print(f"OpenCV Version:           {cv2.__version__}")
print(f"NumPy Version:            {np.__version__}")
print(f"YuNet API (FaceDetectorYN):   {hasattr(cv2, 'FaceDetectorYN')}")
print(f"SFace API (FaceRecognizerSF): {hasattr(cv2, 'FaceRecognizerSF')}")

detector_path = PROJECT_ROOT / "models" / "face_detection_yunet_2023mar.onnx"
verifier_path = PROJECT_ROOT / "models" / "face_recognition_sface_2021dec.onnx"

detector = FaceDetector(model_path=detector_path)
verifier = FaceVerifier(detector=detector, recognizer_model_path=verifier_path)
presence_analyzer = FacePresenceAnalyzer(detector=detector)
print("\nModels initialized successfully on CPU backend!")

ModuleNotFoundError: No module named 'src'

## 2. Load Sample Images

In [ ]:
samples_dir = PROJECT_ROOT / "data" / "samples"
sample_identities = sorted([p for p in samples_dir.iterdir() if p.is_dir() and not p.name.startswith(".")])

print(f"Identities available: {[p.name for p in sample_identities]}")

# Load sample images for Colin Powell and George W Bush
img_cp1 = cv2.imread(str(samples_dir / "Colin_Powell" / "Colin_Powell_0001.jpg"))
img_cp2 = cv2.imread(str(samples_dir / "Colin_Powell" / "Colin_Powell_0002.jpg"))
img_gwb1 = cv2.imread(str(samples_dir / "George_W_Bush" / "George_W_Bush_0001.jpg"))

print(f"Image shapes: CP1: {img_cp1.shape}, CP2: {img_cp2.shape}, GWB1: {img_gwb1.shape}")

## 3. Face Presence & Landmark Extraction (YuNet)

In [ ]:
presence_cp1 = presence_analyzer.analyze(img_cp1)
print(f"Status:           {presence_cp1.status.value}")
print(f"Detected faces:   {presence_cp1.face_count}")
print(f"Inference time:   {presence_cp1.inference_time_ms:.2f} ms")
if presence_cp1.face_count > 0:
    face = presence_cp1.faces[0]
    print(f"  Bounding Box (x, y, w, h): {face.bbox}")
    print(f"  Confidence Score:         {face.confidence:.4f}")
    print(f"  Landmarks (5 points):     {face.landmarks}")

## 4. Single-Pair Face Verification (Positive vs Negative Pairs)

In [ ]:
# Positive Pair: Colin Powell #1 vs Colin Powell #2
pos_result = verifier.verify(img_cp1, img_cp2, metric="cosine")
print("=== Positive Pair (Same Identity) ===")
print(f"Similarity:  {pos_result.similarity:.4f}")
print(f"Threshold:   {pos_result.threshold:.4f}")
print(f"Decision:    {'SAME PERSON' if pos_result.same_person else 'DIFFERENT PERSON'}")
print(f"Latency:     {pos_result.timing_ms.get('total_inference_ms', 0):.2f} ms\n")

# Negative Pair: Colin Powell #1 vs George W Bush #1
neg_result = verifier.verify(img_cp1, img_gwb1, metric="cosine")
print("=== Negative Pair (Different Identities) ===")
print(f"Similarity:  {neg_result.similarity:.4f}")
print(f"Threshold:   {neg_result.threshold:.4f}")
print(f"Decision:    {'SAME PERSON' if neg_result.same_person else 'DIFFERENT PERSON'}")
print(f"Latency:     {neg_result.timing_ms.get('total_inference_ms', 0):.2f} ms")

## 5. Threshold Calibration & Evaluation Benchmark

In [ ]:
from scripts.evaluate_face_verification import build_image_pairs_from_directory, compute_metrics

pairs = build_image_pairs_from_directory(samples_dir)
print(f"Constructed {len(pairs)} pairs for validation.")
if len(pairs) == 0:
    print("Generating sample test media...")
    from scripts.generate_synthetic_test_media import generate_synthetic_media
    generate_synthetic_media()
    pairs = build_image_pairs_from_directory(samples_dir)

scores = []
labels = []
for pair in pairs:
    ia = cv2.imread(str(pair.image_a_path))
    ib = cv2.imread(str(pair.image_b_path))
    res = verifier.verify(ia, ib, metric="cosine")
    scores.append(res.similarity if res.success else None)
    labels.append(pair.is_same_person)

# Sweep thresholds
print("\nThreshold Sweep (Cosine Similarity):")
print("Thresh | Accuracy | Precision | Recall | F1-Score | TP | TN | FP | FN")
print("-" * 68)
for t in np.linspace(0.20, 0.55, 15):
    m = compute_metrics(scores, labels, t, metric="cosine")
    print(f"{m.threshold:6.3f} | {m.accuracy*100:7.1f}% | {m.precision*100:8.1f}% | {m.recall*100:5.1f}% | {m.f1_score:7.3f}  | {m.tp:2d} | {m.tn:2d} | {m.fp:2d} | {m.fn:2d}")